# **Projet Sciences des données :**<br> Système intelligent de prévision des ventes en grande distribution

### Dataset : **Walmart Sales Forecast**

**Problématique :**<br>
Comment concevoir un système intelligent capable de produire des prévisions fiables et exploitables des ventes hebdomadaires  
par magasin et par département afin d'améliorer la prise de décision opérationnelle ?


**Objectif général :**<br>
Développer un modèle prédictif robuste permettant d’anticiper les ventes hebdomadaires avec un haut niveau de précision.

**Objectifs spécifiques :**
- Analyser les facteurs influençant les ventes
- Construire un pipeline de données fiable
- Tester plusieurs modèles de Machine Learning
- Évaluer les performances
- Déployer un prototype utilisable par les décideurs

## 1. Compréhension des données

### Chargement des données

In [33]:
# Importation des bibliothèques nécessaires
# Importation des bibliothèques nécessaires
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [34]:
# Chargement des des données
features = pd.read_csv("Walmart/features.csv")
stores = pd.read_csv("Walmart/stores.csv")
train = pd.read_csv("Walmart/train.csv")
test = pd.read_csv("Walmart/test.csv")

display(features.head())
display(stores.head())
display(train.head())
display(test.head())

,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2010-02-05,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,2010-02-12,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,2010-02-19,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,2010-02-26,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,2010-03-05,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False


### Fusion des jeux de données

In [35]:
# Datasets Features et Stores
stores_features = pd.merge(features, stores, on='Store', how='left') # Jointure gauche
stores_features

,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday,Type,Size
0,1,2010-02-05,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False,A,151315
1,1,2010-02-12,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True,A,151315
2,1,2010-02-19,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False,A,151315
3,1,2010-02-26,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False,A,151315
4,1,2010-03-05,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False,A,151315
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8185,45,2013-06-28,76.05,3.639,4842.29,975.03,3.00,2449.97,3169.69,NaN,NaN,False,B,118221
8186,45,2013-07-05,77.50,3.614,9090.48,2268.58,582.74,5797.47,1514.93,NaN,NaN,False,B,118221
8187,45,2013-07-12,79.37,3.614,3789.94,1827.31,85.72,744.84,2150.36,NaN,NaN,False,B,118221
8188,45,2013-07-19,82.84,3.737,2961.49,1047.07,204.19,363.00,1059.46,NaN,NaN,False,B,118221


In [36]:
# Datasets train et stores_features
df = pd.merge(train, stores_features, on=['Store', 'Date'], how='left')
df = df.drop(columns=['IsHoliday_y'])
df.rename(columns={'IsHoliday_x': 'IsHoliday'}, inplace=True)
df

,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315
2,1,1,2010-02-19,41595.55,False,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315
3,1,1,2010-02-26,19403.54,False,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315
4,1,1,2010-03-05,21827.90,False,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421565,45,98,2012-09-28,508.37,False,64.88,3.997,4556.61,20.64,1.50,1601.01,3288.25,192.013558,8.684,B,118221
421566,45,98,2012-10-05,628.10,False,64.89,3.985,5046.74,NaN,18.82,2253.43,2340.01,192.170412,8.667,B,118221
421567,45,98,2012-10-12,1061.02,False,54.47,4.000,1956.28,NaN,7.89,599.32,3990.54,192.327265,8.667,B,118221
421568,45,98,2012-10-19,760.01,False,56.47,3.969,2004.02,NaN,3.18,437.73,1537.49,192.330854,8.667,B,118221


In [37]:
# Datasets test et stores_features
df_evaluation = pd.merge(test, stores_features, on=['Store', 'Date'], how='left')
df_evaluation.drop('IsHoliday_y', axis=1, inplace=True)
df_evaluation.rename(columns={'IsHoliday_x': 'IsHoliday'}, inplace=True)
df_evaluation.head()

,Store,Dept,Date,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,1,2012-11-02,False,55.32,3.386,6766.44,5147.70,50.82,3639.90,2737.42,223.462779,6.573,A,151315
1,1,1,2012-11-09,False,61.24,3.314,11421.32,3370.89,40.28,4646.79,6154.16,223.481307,6.573,A,151315
2,1,1,2012-11-16,False,52.92,3.252,9696.28,292.10,103.78,1133.15,6612.69,223.512911,6.573,A,151315
3,1,1,2012-11-23,True,56.23,3.211,883.59,4.17,74910.32,209.91,303.32,223.561947,6.573,A,151315
4,1,1,2012-11-30,False,52.34,3.207,2460.03,NaN,3838.35,150.57,6966.34,223.610984,6.573,A,151315


###	Vérification des problèmes de qualité de données

In [38]:
# Vérification des doublons
df.duplicated().sum()

np.int64(0)

In [39]:
# Vérification des valeurs manquantes (en pourcentage)
(df.isnull().mean() * 100).round(2)

Store            0.00
Dept             0.00
Date             0.00
Weekly_Sales     0.00
IsHoliday        0.00
Temperature      0.00
Fuel_Price       0.00
MarkDown1       64.26
MarkDown2       73.61
MarkDown3       67.48
MarkDown4       67.98
MarkDown5       64.08
CPI              0.00
Unemployment     0.00
Type             0.00
Size             0.00
dtype: float64

In [40]:
# Séparation des variables numériques et catégorielles
num_col = df.select_dtypes(include=['int64', 'float64']).columns
cat_col = df.select_dtypes(exclude="number").columns

#cat_col = cat_col.drop('Date', errors='ignore') # isolation de la variable Date

In [41]:
# Vérification des valeurs aberrantes ou outliers

# Méthode de boxplot :
#df.boxplot(figsize=(16,5))
#plt.show()

# Méthode IQR :
Q1 = df[num_col].quantile(0.25)
Q3 = df[num_col].quantile(0.75)
IQR = Q3 - Q1
outliers = (df[num_col] < (Q1 - 1.5 * IQR)) | (df[num_col] > (Q3 + 1.5 * IQR))

outliers_count = outliers.sum() # compte les True par colonne
outliers_percent = (outliers.sum() / len(df)) * 100
print(outliers_percent.round(2))

Store           0.00
Dept            0.00
Weekly_Sales    8.43
Temperature     0.02
Fuel_Price      0.00
MarkDown1       2.30
MarkDown2       4.18
MarkDown3       4.43
MarkDown4       3.06
MarkDown5       1.85
CPI             0.00
Unemployment    7.62
Size            0.00
dtype: float64


In [ ]:
# Vérification des incohérences
for col in df.columns:
    print(df[col].value_counts(dropna=False).sort_index(), "\n")

## **2. Préparer les données**

## 2.2. Préparation des données

### Création de nouvelles variables (Feature Engineering)

In [55]:
# Variable Weekly_Sales_log : afin de  les Valeurs aberrantes de Weekly_Sales (8.43%) en applicant le logarithme
df['Weekly_Sales_log'] = np.log1p(df['Weekly_Sales'])

c:\Users\rosli\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\rosli\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [44]:
# Variables dérivées de Date
df['Date'] = pd.to_datetime(df['Date'])

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Week'] = df['Date'].dt.isocalendar().week    # retourne le numéro de semaine ISO (1–52 ou 53)
#df['Day'] = df['Date'].dt.day
#df['DayOfWeek'] = df['Date'].dt.dayofweek     # 0 = lundi, 6 = dimanche
#df['DayOfYear'] = df['Date'].dt.dayofyear

df.head()

#for col in ['Year', 'Month', 'Week']:
#    print(df[col].value_counts(dropna=False).sort_index(),"\n")

,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size,Weekly_Sales_log,Year,Month,Week
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315,10.123647,2010,2,5
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315,10.737277,2010,2,6
2,1,1,2010-02-19,41595.55,False,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315,10.635773,2010,2,7
3,1,1,2010-02-26,19403.54,False,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315,9.873262,2010,2,8
4,1,1,2010-03-05,21827.90,False,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315,9.990990,2010,3,9


Dans la norme ISO :
La semaine 1 n’est pas forcément celle du 1er janvier, il s'agit de la première semaine qui contient un jeudi.
Conséquence :
- certaines dates de fin décembre peuvent appartenir à la semaine 1 de l’année suivante
- certaines dates de début janvier peuvent appartenir à la dernière semaine de l’année précédente

Les semaines ISO permettent :
- d’avoir des semaines complètes et cohérentes
- d’éviter les semaines coupées au début ou à la fin de l’année
- d’être compatible avec les standards internationaux

In [45]:
# Variable Flag indicant l'existence d'une promotion (Markdown 1 ou 2 ou 3 ou 4 ou 5) pour une semaine donnée
df['Flag'] = np.where(df[[col for col in df.columns if 'MarkDown' in col]].sum(axis=1) != 0, 1, 0)

df['Flag'].value_counts()

Flag
0    270138
1    151432
Name: count, dtype: int64

Mesure de performance en moins avec ce traitement

In [46]:
# Lag de la variable cible par Store & Dept pour les 1 à 4 semaines précédentes
df['Lag_1'] = df.groupby(['Store','Dept'])['Weekly_Sales'].shift(1)
df['Lag_2'] = df.groupby(['Store','Dept'])['Weekly_Sales'].shift(2)
df['Lag_3'] = df.groupby(['Store','Dept'])['Weekly_Sales'].shift(3)
df['Lag_4'] = df.groupby(['Store','Dept'])['Weekly_Sales'].shift(4)

In [47]:
# Gestion des valeurs manquantes créées par les lags
Lags  = ['Lag_1', 'Lag_2', 'Lag_3', 'Lag_4']
df[Lags] = df[Lags].fillna(0)

df.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,...,Size,Weekly_Sales_log,Year,Month,Week,Flag,Lag_1,Lag_2,Lag_3,Lag_4
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,...,151315,10.123647,2010,2,5,0,0.00,0.00,0.00,0.0
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,...,151315,10.737277,2010,2,6,0,24924.50,0.00,0.00,0.0
2,1,1,2010-02-19,41595.55,False,39.93,2.514,NaN,NaN,NaN,...,151315,10.635773,2010,2,7,0,46039.49,24924.50,0.00,0.0
3,1,1,2010-02-26,19403.54,False,46.63,2.561,NaN,NaN,NaN,...,151315,9.873262,2010,2,8,0,41595.55,46039.49,24924.50,0.0
4,1,1,2010-03-05,21827.90,False,46.50,2.625,NaN,NaN,NaN,...,151315,9.990990,2010,3,9,0,19403.54,41595.55,46039.49,24924.5


In [48]:
# Créer une variable Target encoding correspondant la moyenne par catégorie de store et de département pour la variable cible (ventes)